# 00 — Chuẩn hóa dữ liệu (TV1, Việc 1 + Việc 2)

**Add Input:** `shahliza27/ur-fall-detection-dataset`, `minhquang2701/urfall-labels`, `tuyenldvn/falldataset-imvia`. Bật **Internet**, không cần GPU.

Trước khi chạy: điền `version` của 3 dataset vào `configs/data.yaml` (mục `dataset_sources`) để `DATA_CARD.md` ghi đúng.

Đầu ra nằm ở `/kaggle/working/nhom05-falldet-processed`, dùng để tạo Kaggle Dataset **`nhom05-falldet-processed`**.

In [ ]:
# Ô đầu tiên: lấy mã nguồn, cố định seed, in version Python, torch, ultralytics, opencv và version dataset.
# Cần bật Internet để clone repo.
import os, sys
BRANCH = "tv1/gk-data-eda-m1"
REPO = "/kaggle/working/FallDetection"
if not os.path.exists(REPO):
    !git clone --depth 1 -b {BRANCH} https://github.com/ducqwang/FallDetection.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
from src.data.env_info import setup_notebook
setup_notebook("configs/data.yaml", seed=42)

In [ ]:
!ffmpeg -version | head -1
!ffprobe -version | head -1

## Việc 1: chuẩn hóa

Script quét và đối chiếu toàn bộ số liệu với mục 2.1, 2.2 **trước khi** chuyển video. Có chỗ lệch thì dừng và in từng chỗ: khi đó báo TV1, không sửa số cho khớp.

In [ ]:
!python -m src.data.prepare_data --config configs/data.yaml --out-root /kaggle/working/nhom05-falldet-processed --workers 4

## Việc 2: chia tập

In [ ]:
!python -m src.data.split --meta /kaggle/working/nhom05-falldet-processed/metadata.csv --out /kaggle/working/nhom05-falldet-processed/splits.json --seed 42

## Kiểm tra đầu ra

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
OUT = Path("/kaggle/working/nhom05-falldet-processed")
meta = pd.read_csv(OUT / "metadata.csv", dtype={"scene_part": str})
print(meta.groupby(["dataset", "scene", "label"], dropna=False).size())
print("mp4:", len(list((OUT / "videos").glob("*.mp4"))), "| dung lượng (GB):", round(sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file()) / 1e9, 2))
display(Markdown((OUT / "DATA_CARD.md").read_text(encoding="utf-8")))

## Việc tiếp theo

1. Tạo Kaggle Dataset `nhom05-falldet-processed` từ thư mục đầu ra (nút *New Dataset* trong tab Output, hoặc lưu version notebook).
2. Tải `metadata.csv`, `splits.json`, `DATA_CARD.md` về và commit lên nhánh (file nhỏ, được phép commit).